# Universal Knowledge Distillation
Questo notebook implementa un approccio di Knowledge Distillation ibrido. Combina l'efficienza temporale dell'approccio offline tramite `SFTTrainer` (usato in `Summarization.ipynb`) con le funzionalità di profilazione hardware avanzata (Latenza, VRAM, Perplexity) e grafici di `KD_Project.ipynb`.

## 1. Setup e Installazioni

In [ ]:
!pip install -q transformers datasets trl peft evaluate rouge_score bert_score matplotlib bitsandbytes accelerate

## 2. Configurazione
Imposta qui i parametri principali dell'esperimento.

In [ ]:
import os

# --- CONFIGURAZIONE PRINCIPALE ---
# TASK_TYPE: 'summarization' (SAMSum dataset) oppure 'qa' (SQuAD v2 dataset)
TASK_TYPE = "qa" 

# TEACHER_MODEL_ID: 
# Opzione 1: "TinyLlama/TinyLlama-1.1B-Chat-v1.0" (Più veloce, ottimo per test/samsum)
# Opzione 2: "Qwen/Qwen2.5-1.5B-Instruct" (Migliore su QA, richiede quantizzazione 4bit per GPU piccole)
TEACHER_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Flag per decidere se caricare il Teacher in 4-bit (risparmio VRAM)
QUANTIZE_TEACHER = False

# STUDENT_MODEL_ID: Fisso a SmolLM per questo esperimento
STUDENT_MODEL_ID = "HuggingFaceTB/SmolLM-135M"

# --- PARAMETRI DI ADDESTRAMENTO E DEBUG ---
# Limite sample per velocizzare l'esperimento. Imposta a None per usare tutto il dataset.
MAX_TRAIN_SAMPLES = 'all'  # Usa 'all' per tutto il dataset, o un intero (es. 500)
MAX_TEST_SAMPLES = 'all'

STUDENT_BATCH_SIZE = 32
TEACHER_BATCH_SIZE = 8
GRAD_ACCUMULATION = 1
EPOCHS = 3
LEARNING_RATE = 1e-5
MAX_SEQ_LENGTH = 1024

OUTPUT_DIR = f"./student_distilled_{TASK_TYPE}"
TEACHER_DATASET_DIR = f"./dataset_distilled_{TASK_TYPE}"


## 3. Preparazione dei Dati
Caricamento logico del dataset e formattazione basata sul task scelto.

In [ ]:
from datasets import load_dataset
import random

print(f"Caricamento dataset per il task: {TASK_TYPE}")

if TASK_TYPE == "summarization":
    dataset = load_dataset("knkarthick/samsum")
    def format_teacher_prompt(example):
        messages = [
            {"role": "system", "content": "You are a highly accurate summarization assistant. Provide a concise summary of the following conversation."},
            {"role": "user", "content": f"Summarize this dialogue:\n\n{example['dialogue']}"}
        ]
        return {"teacher_prompt_messages": messages, "target": example['summary'], "input_text": example['dialogue']}

elif TASK_TYPE == "qa":
    dataset = load_dataset("databricks/databricks-dolly-15k")
    # Dolly ha solo la partizione 'train', facciamo uno split 90/10 manuale fissando il seed
    dataset = dataset['train'].train_test_split(test_size=0.1, seed=42)
    
    def format_teacher_prompt(example):
        instruction = example['instruction']
        context = example['context']
        if context and context.strip():
            user_content = f"Context: {context}\n\nInstruction: {instruction}"
            input_text = f"Context: {context[:300]}...\nInstruction: {instruction}"
        else:
            user_content = f"Instruction: {instruction}"
            input_text = f"Instruction: {instruction}"
            
        messages = [
            {"role": "system", "content": "You are a helpful and concise AI assistant. Follow the user's instruction."},
            {"role": "user", "content": user_content}
        ]
        return {"teacher_prompt_messages": messages, "target": example['response'], "input_text": input_text}
else:
    raise ValueError("TASK_TYPE non supportato. Scegli 'summarization' o 'qa'.")

train_data = dataset['train']
test_data = dataset['test']

if MAX_TRAIN_SAMPLES != 'all':
    train_data = train_data.select(range(min(int(MAX_TRAIN_SAMPLES), len(train_data))))
if MAX_TEST_SAMPLES != 'all':
    test_data = test_data.select(range(min(int(MAX_TEST_SAMPLES), len(test_data))))

train_data = train_data.map(format_teacher_prompt)
test_data = test_data.map(format_teacher_prompt)

print(f"Train samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")


## 4. Generazione Offline (Teacher)
Usiamo il Teacher per generare l'output ideale da far imitare allo Student. Le risposte vengono salvate su disco.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.auto import tqdm

print(f"Inizializzazione Teacher: {TEACHER_MODEL_ID}")

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
# Fondamentale per la generazione in batch su modelli CausalLM
teacher_tokenizer.padding_side = 'left'

if QUANTIZE_TEACHER:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    teacher_model = AutoModelForCausalLM.from_pretrained(
        TEACHER_MODEL_ID, device_map="auto", quantization_config=quantization_config
    )
else:
    teacher_model = AutoModelForCausalLM.from_pretrained(
        TEACHER_MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
    )

teacher_model.eval()

if os.path.exists(TEACHER_DATASET_DIR):
    print("Dataset distillato trovato! Caricamento da disco...")
    from datasets import load_from_disk
    distilled_train = load_from_disk(TEACHER_DATASET_DIR)
else:
    print("Generazione offline delle risposte del Teacher...")
    teacher_responses = []
    
    for i in tqdm(range(0, len(train_data), TEACHER_BATCH_SIZE), desc="Teacher Generation (Batched)"):
        batch = train_data[i:i+TEACHER_BATCH_SIZE]
        prompt_strs = [teacher_tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch["teacher_prompt_messages"]]
        
        inputs = teacher_tokenizer(prompt_strs, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(teacher_model.device)
        
        with torch.no_grad():
            outputs = teacher_model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=teacher_tokenizer.pad_token_id)
            
        generated_tokens = outputs[:, inputs['input_ids'].shape[1]:]
        gen_texts = teacher_tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        teacher_responses.extend([t.strip() for t in gen_texts])
        
    distilled_train = train_data.add_column("teacher_response", teacher_responses)
    distilled_train.save_to_disk(TEACHER_DATASET_DIR)
    
print("Generazione offline completata.")

del teacher_model
torch.cuda.empty_cache()


## 5. Addestramento dello Student
Usiamo TRL `SFTTrainer` per effettuare il fine-tuning.

In [ ]:
from trl import SFTTrainer, SFTConfig

print(f"Inizializzazione Student: {STUDENT_MODEL_ID}")

student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID)
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token
    
if student_tokenizer.chat_template is None:
    student_tokenizer.chat_template = (
        "{% for message in messages %}"
        "<|im_start|>{{ message['role'] }}\n"
        "{{ message['content'] }}<|im_end|>\n"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "<|im_start|>assistant\n"
        "{% endif %}"
    )

student_model = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")

def format_to_prompt_completion(example):
    messages = list(example['teacher_prompt_messages'])
    messages.append({"role": "assistant", "content": example['teacher_response']})
    return {"messages": messages}

pc_dataset = distilled_train.map(format_to_prompt_completion, remove_columns=distilled_train.column_names)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=STUDENT_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=10,
    num_train_epochs=EPOCHS,
    bf16=True,
    optim="adamw_torch_fused",
    report_to="none",
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="messages",
)

trainer = SFTTrainer(
    model=student_model,
    args=training_args,
    train_dataset=pc_dataset,
    processing_class=student_tokenizer,
)

print("Avvio addestramento...")
trainer.train()

FINAL_OUTPUT_DIR = f"{OUTPUT_DIR}_final"
trainer.save_model(FINAL_OUTPUT_DIR)
student_tokenizer.save_pretrained(FINAL_OUTPUT_DIR)
print(f"Modello salvato in {FINAL_OUTPUT_DIR}")


## 6. Valutazione e Profilazione Hardware
Calcolo delle metriche qualitative (ROUGE, BERTScore) e hardware (Latenza, Perplexity, Parametri).

In [ ]:
import evaluate
import time
import math
import numpy as np
from tqdm.auto import tqdm

rouge_metric = evaluate.load("rouge")
bert_metric = evaluate.load("bertscore")

def profile_and_evaluate(model_id, is_baseline=False):
    print(f"Valutazione modello: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if tokenizer.chat_template is None:
        tokenizer.chat_template = student_tokenizer.chat_template

    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32, device_map="auto")
    model.eval()
    
    predictions = []
    references = []
    
    total_time = 0
    total_generated_tokens = 0
    total_loss = 0.0
    
    for sample in tqdm(test_data, desc="Evaluating"):
        prompt_str = tokenizer.apply_chat_template(sample["teacher_prompt_messages"], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_str, return_tensors="pt", padding=True).to(model.device)
        
        full_text = prompt_str + sample['target'] + tokenizer.eos_token
        full_inputs = tokenizer(full_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            loss_output = model(**full_inputs, labels=full_inputs["input_ids"])
            total_loss += loss_output.loss.item()
            
            start_time = time.time()
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
            end_time = time.time()
            
        generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        gen_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        
        total_time += (end_time - start_time)
        total_generated_tokens += len(generated_tokens)
        
        predictions.append(gen_text)
        references.append(sample['target'])
        
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    bert_results = bert_metric.compute(predictions=predictions, references=references, lang="en")
    mean_bert_f1 = sum(bert_results['f1']) / len(bert_results['f1'])
    
    avg_loss = total_loss / len(test_data)
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float('inf')
    ms_per_token = (total_time * 1000) / total_generated_tokens if total_generated_tokens > 0 else 0
    param_count = sum(p.numel() for p in model.parameters()) / 1e6
    
    del model
    torch.cuda.empty_cache()
    
    return {
        "Perplexity": perplexity,
        "ROUGE-L": rouge_results['rougeL'],
        "BERTScore-F1": mean_bert_f1,
        "Latency/Token (ms)": ms_per_token,
        "Parameters (M)": param_count,
        "predictions": predictions
    }

res_distilled = profile_and_evaluate(FINAL_OUTPUT_DIR)
res_baseline = profile_and_evaluate(STUDENT_MODEL_ID, is_baseline=True)

print("\n=== RISULTATI COMPARATIVI ===")
print("Student Baseline (Zero-Shot):")
for k, v in res_baseline.items():
    if k != "predictions": print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
        
print("\nStudent Distilled:")
for k, v in res_distilled.items():
    if k != "predictions": print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


## 7. Grafici per Relazione
Rappresentazione visiva delle metriche da esportare nei report.

In [ ]:
import matplotlib.pyplot as plt

metrics = ['Perplexity', 'ROUGE-L', 'BERTScore-F1', 'Latency/Token (ms)']
baseline_vals = [res_baseline[m] for m in metrics]
distilled_vals = [res_distilled[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, baseline_vals, width, label='Student Baseline ZS', color='lightcoral')
rects2 = ax.bar(x + width/2, distilled_vals, width, label='Student Distilled', color='mediumseagreen')

ax.set_ylabel('Scores / Valori')
ax.set_title('Confronto Prestazioni: Baseline vs Distilled')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

fig.tight_layout()
plt.show()


## 8. Ispezione Qualitativa (Esempi Finali)
Verifichiamo manualmente la qualità dell'output su alcuni esempi casuali dal test set.

In [ ]:
print("=== ISPEZIONE QUALITATIVA DEGLI OUTPUT ===\n")
import random

indices = random.sample(range(len(test_data)), min(3, len(test_data)))

for i, idx in enumerate(indices, 1):
    sample = test_data[idx]
    pred_baseline = res_baseline['predictions'][idx]
    pred_distilled = res_distilled['predictions'][idx]
    
    print(f"--- SAMPLE {i} ---")
    print(f"INPUT:\n{sample['input_text'].strip()}\n")
    print(f"TARGET IDEALE (Human):\n{sample['target'].strip()}\n")
    print(f"STUDENT BASELINE (Zero-Shot):\n{pred_baseline.strip()}\n")
    print(f"STUDENT DISTILLED:\n{pred_distilled.strip()}\n")
    print("="*80 + "\n")
